# Person 2 — Step 2: Download Thumbnails

Downloads high-res thumbnails from `thumbnail_url_high` with:
- Concurrent requests (fast)
- Skip-if-exists (safe to re-run)
- Progress bar
- A download manifest CSV for downstream notebooks

In [ ]:
import sys
sys.path.append('../../')

import pandas as pd
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

from shared.config import VIDEOS_CSV, CHANNELS_CSV, THUMBNAILS_DIR, PERSON2_DIR

In [ ]:
videos   = pd.read_csv(VIDEOS_CSV)
channels = pd.read_csv(CHANNELS_CSV)[['channel_id', 'channel_title', 'group_label']]
df = videos.merge(channels, on='channel_id')

# Only rows with a thumbnail URL
df = df.dropna(subset=['thumbnail_url_high']).copy()
print(f"{len(df)} videos with thumbnail URLs")

In [ ]:
def download_one(row):
    video_id = row['video_id']
    url      = row['thumbnail_url_high']
    dest     = THUMBNAILS_DIR / f"{video_id}.jpg"

    if dest.exists():
        return video_id, 'skipped'

    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        dest.write_bytes(r.content)
        return video_id, 'ok'
    except Exception as e:
        return video_id, f'error: {e}'

rows = df.to_dict('records')
results = {}

with ThreadPoolExecutor(max_workers=20) as pool:
    futures = {pool.submit(download_one, row): row['video_id'] for row in rows}
    for future in tqdm(as_completed(futures), total=len(futures), desc='Downloading thumbnails'):
        vid, status = future.result()
        results[vid] = status

status_series = pd.Series(results).value_counts()
print(status_series)

In [ ]:
# Save manifest so downstream notebooks know which files exist
manifest = (
    df[['video_id', 'channel_title', 'group_label', 'view_count', 'like_count', 'comment_count', 'duration_seconds', 'published_at']]
    .assign(
        download_status=lambda d: d['video_id'].map(results),
        thumb_path=lambda d: d['video_id'].apply(lambda v: str(THUMBNAILS_DIR / f"{v}.jpg"))
    )
)
manifest_path = PERSON2_DIR / 'outputs' / 'thumbnail_manifest.csv'
manifest.to_csv(manifest_path, index=False)
print(f"Manifest saved → {manifest_path}")
manifest[manifest['download_status'] == 'ok'].shape